# §2.1.4 — MLE의 표집 분포를 직접 재 보기

> 딥러닝 교재 · 1부 2장 1절 4항 (🐍)
> 선행: §2.1.1(로그우도의 합 분해) · §2.1.2(MLE = KL 최소화) · §2.1.3(일치성 · 점근 정규성 · 크라메르–라오)

## 이 노트북이 답하는 질문

1. **점근 정규성이 정말 성립하는가?** $\hat\theta$ 의 표집 분포를 직접 그려 정규 곡선과 대조한다.
2. **분산이 피셔 정보의 역수에 접근하는가?** $n\cdot\mathrm{Var}(\hat\theta)$ 를 훑어 $I(\theta^{\ast})^{-1}$ 에 수렴하는지 본다.
3. **$n$ 이 얼마나 커야 하는가?** §2.1.3이 침묵한 질문이다. 모형마다 답이 다르다.
4. **정칙 조건이 깨지면 무엇이 달라지는가?** §2.1.3의 R4를 위반하는 모형을 대조군으로 둔다.

**예상 실행 시간** CPU 단일 코어 약 25초 (`FAST = True`이면 약 10초).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False
SEED     = 20260805
R        = 40_000     # 시행 수 (표집 분포를 그릴 표본 수)
P_TRUE   = 0.3        # 베르누이 참 모수
LAM_TRUE = 1.0        # 지수분포 참 모수 (rate)
TH_TRUE  = 1.0        # 균등분포 참 모수
SAVE_PDF = False
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────
if FAST:
    R = 10_000

CB = ['#000000','#E69F00','#56B4E9','#009E73','#D55E00','#0072B2','#CC79A7','#F0E442']
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3,
                     'axes.prop_cycle':plt.cycler(color=CB),'figure.autolayout':True})
import matplotlib.font_manager as fm
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR',
                            'Noto Sans KR','NanumBarunGothic','Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en
_fi=[0]
def show(name):
    _fi[0] += 1
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        plt.savefig(os.path.join(FIG_DIR, f'fig_2_1_4_{_fi[0]}_{name}.pdf'),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

print(f"numpy {np.__version__} | FAST={FAST} | R={R:,} | 한글폰트: {KO_FONT or '없음(영문 라벨)'}")

---
## 1. 세 모형과 표집 요령

세 모형을 쓴다. 앞의 둘은 §2.1.3의 정칙 조건을 만족하고, **셋째는 R4를 위반한다.**

| | 모형 | MLE | $I(\theta^{\ast})^{-1}$ | 정칙 조건 |
|---|---|---|---|---|
| A | $\mathrm{Bernoulli}(p)$ | $\hat p = \bar z$ | $p(1-p)$ | 만족 |
| B | $\mathrm{Exponential}(\lambda)$, 밀도 $\lambda e^{-\lambda z}$ | $\hat\lambda = 1/\bar z$ | $\lambda^2$ | 만족 |
| C | $\mathrm{Uniform}(0,\theta)$ | $\hat\theta = \max_i z_i$ | — | **R4 위반** (지지집합이 $\theta$ 에 의존) |

### 표집 요령 — 왜 $n$ 을 아주 크게 할 수 있는가

$\hat\theta$ 의 분포를 알고 싶은데, 순진하게 하면 시행마다 $n$ 개를 뽑아야 하므로 $R \times n$ 개가 필요하다.
그런데 **세 경우 모두 충분통계량의 분포가 폐형식으로 알려져 있으므로 $\hat\theta$ 를 직접 뽑을 수 있다.**

$$\textstyle\sum_i z_i \sim \mathrm{Binomial}(n, p) \qquad
\textstyle\sum_i z_i \sim \mathrm{Gamma}(n, 1/\lambda) \qquad
\max_i z_i \overset{d}{=} \theta \cdot U^{1/n},\ U \sim \mathrm{Uniform}(0,1)$$

시행 하나에 난수 **한 개**면 된다. 그래서 $n = 10^5$ 까지 훑어도 비용이 같다.
다만 요령이 맞는지 먼저 확인한다.

In [ ]:
def mle_bern(n, r, g):  return g.binomial(n, P_TRUE, size=r) / n
def mle_expo(n, r, g):  return 1.0 / (g.gamma(n, 1.0/LAM_TRUE, size=r) / n)
def mle_unif(n, r, g):  return TH_TRUE * g.uniform(0, 1, size=r)**(1.0/n)

MODELS = [
    ('A ' + lab('베르누이','Bernoulli'), mle_bern, P_TRUE,   P_TRUE*(1-P_TRUE), CB[5]),
    ('B ' + lab('지수','Exponential'),   mle_expo, LAM_TRUE, LAM_TRUE**2,       CB[3]),
    ('C ' + lab('균등 (R4 위반)','Uniform (R4 violated)'), mle_unif, TH_TRUE, None, CB[4]),
]

# ── 요령 검증: 직접 표집 vs 무식한 표집 ──
_n, _r = 50, 20_000
_g1, _g2 = np.random.default_rng(1), np.random.default_rng(2)
checks = [
    (lab('베르누이','Bernoulli'), mle_bern(_n,_r,_g1),
     (_g2.random((_r,_n)) < P_TRUE).mean(1)),
    (lab('지수','Exponential'), mle_expo(_n,_r,np.random.default_rng(3)),
     1.0/np.random.default_rng(4).exponential(1/LAM_TRUE, size=(_r,_n)).mean(1)),
    (lab('균등','Uniform'), mle_unif(_n,_r,np.random.default_rng(5)),
     np.random.default_rng(6).uniform(0,TH_TRUE,size=(_r,_n)).max(1)),
]
print(f"표집 요령 검증 (n={_n}, 시행 {_r:,})")
print("             직접 평균 / 무식 평균      직접 분산 / 무식 분산")
for nm, a, b in checks:
    print(f"  {nm:>10}: {a.mean():.5f} / {b.mean():.5f}    {a.var():.6f} / {b.var():.6f}")
print("\n-> 일치. 이하 직접 표집을 쓴다.")

---
## 2. 표집 분포를 그려 보기

§2.1.3 (나)의 주장은 이것이었다.

$$\sqrt{n}\,(\hat\theta_n - \theta^{\ast}) \ \xrightarrow{\ d\ } \ \mathcal{N}\big(0,\ I(\theta^{\ast})^{-1}\big)$$

곧 $\hat\theta_n \approx \mathcal{N}\big(\theta^{\ast},\ I(\theta^{\ast})^{-1}/n\big)$ 이어야 한다. 히스토그램에 그 곡선을 겹쳐 그린다.

In [ ]:
NS_HIST = [5, 50, 1000]
fig, axes = plt.subplots(3, 3, figsize=(10.4, 7.2))
for row, (nm, f, th, Iinv, c) in enumerate(MODELS):
    for col, n in enumerate(NS_HIST):
        ax = axes[row, col]
        s = f(n, R, np.random.default_rng(SEED + 7*n + row))
        ax.hist(s, bins=60, density=True, color=c, alpha=0.55, edgecolor='none')
        if Iinv is not None:
            sd = np.sqrt(Iinv/n)
            xg = np.linspace(s.min(), s.max(), 400)
            ax.plot(xg, stats.norm.pdf(xg, th, sd), color=CB[0], lw=1.5,
                    label=lab('점근 정규','asymptotic normal'))
        ax.axvline(th, color=CB[0], lw=1.0, ls=':')
        ax.set_yticks([])
        if row == 0:
            ax.set_title(f'$n$ = {n}', fontsize=10)
        if col == 0:
            ax.set_ylabel(nm, fontsize=9)
        if row == 0 and col == 0:
            ax.legend(fontsize=7)
axes[2,1].set_xlabel(lab(r'$\hat\theta$', r'$\hat\theta$'))
fig.suptitle(lab('위 두 줄은 정규로 좁혀지고, 아래 줄은 모양이 바뀌지 않는다',
                 'the top two rows converge to normal; the bottom does not'), y=1.02, fontsize=10)
show('sampling_distributions')

> **A와 B는 $n$ 이 커지며 검은 곡선에 붙는다.** C는 다르다 — 좁아지기는 하지만 **모양이 정규가 아니고,
> $\theta^{\ast}$ 를 중심으로 대칭도 아니다.** $\hat\theta = \max_i z_i \le \theta^{\ast}$ 이므로 언제나 왼쪽에 치우친다.
>
> ⚠︎ A의 $n = 50$ 히스토그램이 빗살처럼 보이는 것은 오류가 아니다. $\hat p = \bar z$ 가 $1/n$ 의 배수만
> 취하는 **격자 확률변수**이므로 어떤 $n$ 에서도 연속분포가 되지 못한다. 정규 근사는 격자 간격이 표준편차보다
> 훨씬 작아질 때 쓸 만해지며, 그것이 $n$ 이 커야 하는 또 하나의 이유다.

---
## 3. 분산이 피셔 정보의 역수로 가는가

$n \cdot \mathrm{Var}(\hat\theta_n)$ 을 $n$ 에 대해 훑는다. §2.1.3이 옳다면 $I(\theta^{\ast})^{-1}$ 로 수렴해야 한다.

In [ ]:
NS = [5, 10, 20, 50, 100, 300, 1000, 3000, 10_000] + ([] if FAST else [30_000, 100_000])
var_tab = {}
for nm, f, th, Iinv, c in MODELS:
    var_tab[nm] = np.array([f(n, R, np.random.default_rng(SEED + 11*n)).var() for n in NS])

print("       n        A: n·Var      B: n·Var      C: n·Var      C: n²·Var")
for i, n in enumerate(NS):
    a = n*var_tab[MODELS[0][0]][i]; b = n*var_tab[MODELS[1][0]][i]
    cc = var_tab[MODELS[2][0]][i]
    print(f"{n:>8}   {a:11.5f}   {b:11.5f}   {n*cc:11.5f}   {n*n*cc:11.5f}")
print(f"\n  이론값:  A -> p(1-p) = {P_TRUE*(1-P_TRUE):.5f}    B -> λ² = {LAM_TRUE**2:.5f}")
print("  C는 n·Var이 0으로 가고 n²·Var이 상수로 간다 -> 수렴 속도가 1/√n 이 아니라 1/n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8))
for nm, f, th, Iinv, c in MODELS[:2]:
    axes[0].semilogx(NS, np.array(NS)*var_tab[nm], 'o-', ms=4, color=c, label=nm)
    axes[0].axhline(Iinv, color=c, ls='--', lw=1.0)
axes[0].set_xlabel(lab('표본 수 $n$', 'sample size $n$'))
axes[0].set_ylabel(r'$n \cdot \mathrm{Var}(\hat\theta_n)$')
axes[0].set_ylim(0, 3)
axes[0].set_title(lab('점선이 $I(\\theta^{*})^{-1}$ — 정칙 조건을 만족하는 두 모형',
                      'dashed lines are $I^{-1}$'), fontsize=10)
axes[0].legend(fontsize=8)

nmC = MODELS[2][0]
axes[1].loglog(NS, np.array(NS)*var_tab[nmC], 'o-', ms=4, color=CB[4],
               label=r'$n \cdot \mathrm{Var}$')
axes[1].loglog(NS, np.array(NS)**2*var_tab[nmC], 's-', ms=4, color=CB[1],
               label=r'$n^2 \cdot \mathrm{Var}$')
axes[1].set_xlabel(lab('표본 수 $n$', 'sample size $n$'))
axes[1].set_title(lab('C: 안정되는 것은 $n^2\\cdot$Var 이다',
                      'C: it is $n^2 \\cdot$Var that stabilizes'), fontsize=10)
axes[1].legend(fontsize=8)
show('variance_to_fisher')

> ### 두 주장을 갈라서 읽을 것
>
> **A(베르누이)는 $n = 5$ 에서 이미 $n\cdot\mathrm{Var}$ 이 정확히 $p(1-p)$ 다.** 우연이 아니라
> $\mathrm{Var}(\bar z) = p(1-p)/n$ 이 **모든 $n$ 에서 정확히** 성립하기 때문이다.
> 곧 A에서 점근적인 것은 **분산이 아니라 정규성뿐**이다.
>
> **B(지수)는 위에서 접근한다.** $\hat\lambda = 1/\bar z$ 가 비선형 변환이라 유한 표본에서 분산이 더 크다.
> **두 주장(분산과 정규성)이 서로 다른 속도로 성립한다**는 것을 A와 B의 대비가 보여 준다.
>
> **C는 $1/n$ 속도다.** §2.1.3의 R4(지지집합이 $\theta$ 에 의존하지 않을 것)를 위반하면 결론이 이렇게 달라진다.

---
## 4. $n$ 이 얼마나 커야 하는가

§2.1.3은 이 질문에 침묵한다. §1.7.1의 보편 근사 정리가 폭에 대해 침묵한 것과 같은 형태다.
표집 분포를 표준화한 뒤 정규분포와의 **콜모고로프–스미르노프 거리**로 정규성을 잰다.

In [ ]:
ks_tab = {}
for nm, f, th, Iinv, c in MODELS:
    ks = []
    for n in NS:
        s = f(n, R, np.random.default_rng(SEED + 13*n))
        z = (s - s.mean()) / s.std()                 # 형태만 보기 위해 경험 표준화
        ks.append(stats.kstest(z, 'norm').statistic)
    ks_tab[nm] = np.array(ks)

print("       n          A          B          C")
for i, n in enumerate(NS):
    print(f"{n:>8}   {ks_tab[MODELS[0][0]][i]:8.4f}   "
          f"{ks_tab[MODELS[1][0]][i]:8.4f}   {ks_tab[MODELS[2][0]][i]:8.4f}")

# 유한 시행 수가 만드는 측정 바닥: 참으로 정규인 표본에서도 KS는 이만큼 나온다
KS_FLOOR = 0.87/np.sqrt(R)
_ok = [i for i, n in enumerate(NS) if ks_tab[MODELS[0][0]][i] > 3*KS_FLOOR]
print(f"\n측정 바닥 (시행 {R:,}개에서 참 정규 표본의 기대 KS) = {KS_FLOOR:.4f}")
print(f"바닥의 3배를 넘는 구간 n <= {NS[_ok[-1]]} 에서만 기울기를 적합한다\n")
for nm in [MODELS[0][0], MODELS[1][0]]:
    sl, _ = np.polyfit(np.log(np.array(NS)[_ok]), np.log(ks_tab[nm][_ok]), 1)
    print(f"{nm}: 로그-로그 기울기 {sl:+.3f}  (베리–에센 정리가 예측하는 -1/2)")
print(f"{MODELS[2][0]}: 줄지 않는다 — 정규로 가지 않는다")

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 4.0))
for nm, f, th, Iinv, c in MODELS:
    ax.loglog(NS, ks_tab[nm], 'o-', ms=4, color=c, label=nm)
ax.loglog(NS, 0.5/np.sqrt(NS), 'k:', lw=1.2, label=lab(r'기울기 $-1/2$', r'slope $-1/2$'))
ax.axhline(KS_FLOOR, color=CB[6], lw=1.2, ls='--',
           label=lab(f'측정 바닥 (시행 {R:,})', f'measurement floor (R={R:,})'))
ax.set_xlabel(lab('표본 수 $n$', 'sample size $n$'))
ax.set_ylabel(lab('정규분포와의 KS 거리', 'KS distance to normal'))
ax.set_title(lab('정규 근사의 오차는 $n^{-1/2}$ 로 준다 — 다만 C는 예외',
                 'normal approximation error decays as $n^{-1/2}$ — except C'), fontsize=10)
ax.legend(fontsize=8)
show('normality_rate')

> **정규성은 분산보다 훨씬 느리게 온다.** KS 거리가 $n^{-1/2}$ 로만 줄기 때문에,
> $n = 100$ 에서도 오차가 수 %p 남습니다. **"$n$ 이 크면 정규"라는 말의 '크다'가 얼마인지는 모형이 정합니다.**
>
> 그리고 C는 아무리 커도 줄지 않습니다. 극한 분포가 존재하기는 하지만 정규가 아닙니다 —
> 실제로 $n(\theta^{\ast} - \hat\theta)/\theta^{\ast}$ 가 지수분포로 수렴합니다.

In [ ]:
# C의 참 극한 분포 확인
n_big = 10_000
u = mle_unif(n_big, R, np.random.default_rng(SEED+99))
w = n_big*(TH_TRUE - u)/TH_TRUE
fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.hist(w, bins=80, density=True, color=CB[4], alpha=0.55, edgecolor='none',
        label=lab(r'$n(\theta^{*}-\hat\theta)/\theta^{*}$', r'$n(\theta^{*}-\hat\theta)/\theta^{*}$'))
xg = np.linspace(0, 8, 300)
ax.plot(xg, np.exp(-xg), color=CB[0], lw=1.8, label=lab('지수분포 Exp(1)', 'Exponential(1)'))
ax.set_xlim(0, 8); ax.set_yticks([])
ax.set_title(lab(f'C의 극한 분포는 정규가 아니라 지수다 ($n$={n_big:,})',
                 f'the limit for C is exponential, not normal'), fontsize=10)
ax.legend(fontsize=8)
show('uniform_limit')
print(f"KS 거리 (Exp(1) 대비): {stats.kstest(w, 'expon').statistic:.4f}")

---
## 5. 피셔 정보를 세 가지로 계산해 보기

§2.1.3에서 피셔 정보를 세 가지 방식으로 쓸 수 있다고 했다. 실제로 세 값이 일치하는지 본다.
포아송 모형 $p_\lambda(z) = e^{-\lambda}\lambda^z / z!$ 을 쓴다 ($I(\lambda) = 1/\lambda$).

| 방식 | 식 |
|---|---|
| 해석적 | $I(\lambda^{\ast}) = 1/\lambda^{\ast}$ |
| 스코어 공분산 | $\frac1n\sum_i s_{\hat\lambda}(z_i)^2$ |
| 관측 정보 | $-\frac1n\sum_i \partial_\lambda^2 \log p_{\hat\lambda}(z_i)$ |

In [ ]:
LAM_P = 4.0
print("   n      해석적 1/λ*    스코어 공분산   관측 정보     스코어 평균")
for n in [20, 100, 1000, 10_000] + ([] if FAST else [100_000]):
    z = np.random.default_rng(SEED+3).poisson(LAM_P, size=n).astype(float)
    lh = z.mean()                                  # MLE
    score = z/lh - 1.0                             # ∂_λ log p = z/λ - 1
    I_score = float(np.mean(score**2))
    I_obs   = float(np.mean(z/lh**2))              # -∂²_λ log p = z/λ²
    print(f"{n:>7}   {1.0/LAM_P:12.5f}   {I_score:13.5f}   {I_obs:10.5f}   {np.mean(score):+.2e}")
print("\n-> 세 값이 n이 커지며 수렴한다. 스코어의 평균이 0인 것도 확인된다 (§2.1.3의 보조 결과).")
print("   유한 n에서는 셋이 다르며, 어느 것을 쓸지가 실무에서 실제 선택이다 (§50.2의 라플라스 근사).")

---
## 6. 자기 점검

1. 3절에서 A는 모든 $n$ 에서 $n\cdot\mathrm{Var}$ 이 정확했는데 B는 위에서 접근했다. **왜 다른가?** ($\hat\lambda = 1/\bar z$ 의 형태를 볼 것)
2. B의 $n\cdot\mathrm{Var}$ 이 위에서 접근한다는 것은 유한 표본에서 크라메르–라오 하한을 **위반하지 않는다는** 뜻인가?
3. `P_TRUE` 를 0.02로 바꾸면 A의 정규 수렴이 빨라지겠는가 느려지겠는가? 예측한 뒤 확인하라.
4. C에서 $\hat\theta$ 는 언제나 $\theta^{\ast}$ 보다 **작다.** 이것이 §2.1.5(유한 표본 편향)와 어떻게 이어지는가?

In [ ]:
# 자기 점검 3의 확인 — 참 모수가 경계에 가까우면
print("베르누이: 참 모수에 따른 정규 수렴 속도 (KS 거리)")
print("      n      p=0.30     p=0.02")
for n in [10, 100, 1000, 10_000]:
    row = []
    for p in [0.30, 0.02]:
        s = np.random.default_rng(SEED+17*n).binomial(n, p, size=R)/n
        if s.std() == 0:
            row.append(np.nan); continue
        row.append(stats.kstest((s-s.mean())/s.std(), 'norm').statistic)
    print(f"{n:>7}   {row[0]:8.4f}   {row[1]:8.4f}")
print("\n-> p가 경계에 가까우면 분포가 치우쳐 정규 수렴이 훨씬 느리다.")
print("   §2.1.3의 R2(참 모수가 Θ의 내부에 있을 것)가 '가까워도' 실질적으로 문제가 된다는 뜻이다.")

# 자기 점검 4의 확인 — C의 편향
print("\n균등 모형의 유한 표본 편향 (이론: E[θ̂] = nθ/(n+1))")
for n in [5, 20, 100, 1000]:
    s = mle_unif(n, R, np.random.default_rng(SEED+23*n))
    print(f"  n={n:>5}: 실측 평균 {s.mean():.6f}   이론 {n/(n+1)*TH_TRUE:.6f}   편향 {s.mean()-TH_TRUE:+.6f}")

---
## 7. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `R` | 0절 | 40,000 | 시행 수. 줄이면 히스토그램과 KS 거리가 거칠어진다 |
| `P_TRUE` | 0절 | 0.3 | 0.02로 두면 정규 수렴이 크게 느려진다 (자기 점검 3) |
| `LAM_TRUE` | 0절 | 1.0 | 지수분포의 척도. $I^{-1} = \lambda^2$ 이 함께 움직인다 |
| `NS` | 3절 | 5 ~ 100,000 | 표본 수 격자 |
| `NS_HIST` | 2절 | [5, 50, 1000] | 히스토그램을 그릴 $n$ |
| `LAM_P` | 5절 | 4.0 | 포아송 참 모수 |

**권하는 첫 실험** — `P_TRUE = 0.02` 로 두고 전체를 다시 실행하십시오. $n\cdot\mathrm{Var}$ 은 여전히
**모든 $n$ 에서 정확히** $p(1-p)$ 인데 KS 거리는 훨씬 느리게 줍니다.
**분산이 맞는 것과 정규 근사가 쓸 만한 것은 다른 문제**임을 한 번에 확인하는 방법입니다.

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")